# Phase 2 — Analyse Exploratoire des Données (EDA)
## Images de surveillance du Merapi — 2014–2018

**Objectifs** :
1. Vérifier la complétude et la cohérence de l'index
2. Cartographier la couverture temporelle
3. Inspecter visuellement un échantillon contrôlé
4. Calculer des statistiques image (luminosité, contraste, taille)
5. Formuler les premières observations pour la Phase 3

**Garde-fous** :
- Aucune cellule ne charge plus de 50 images en mémoire
- L'analyse statistique est faite sur des échantillons plafonnés
- Chaque cellule est annotée : `[LÉGER]`, `[MOYEN]` ou `[COÛTEUX]`
- tqdm est utilisé pour toute boucle > 100 itérations

In [ ]:
# [LÉGER] — Imports et configuration
import sys, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# Détection racine projet
NOTEBOOK_DIR = Path(os.getcwd())
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# S'assurer que src/ est importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INDEX_PATH = PROJECT_ROOT / "data" / "index" / "index.csv"

print(f"Projet : {PROJECT_ROOT}")
print(f"Index  : {INDEX_PATH}")
print(f"Raw    : {RAW_DIR}")

---
## A. Vérification du dataset
Cellules purement CSV — **instantanées**.

In [ ]:
# [LÉGER] — Chargement de l'index
df = pd.read_csv(INDEX_PATH)
print(f"Entrées dans l'index : {len(df):,}")
print(f"Colonnes ({len(df.columns)}) : {list(df.columns)}")
df.head(3)

In [ ]:
# [LÉGER] — Typage et nettoyage
int_cols = ["year", "month", "day", "hour", "minute", "second"]
for c in int_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

# Filtrer 2014–2018 uniquement
df = df[df["year"].between(2014, 2018)].copy()
print(f"Entrées 2014–2018 : {len(df):,}")

# Identifier la source (icône vs image publique)
df["source_type"] = df["url"].apply(
    lambda u: "icone" if "/icones/" in str(u)
    else ("image_publique" if "/images_publiques/" in str(u) else "autre")
)

# Extraire la caméra depuis le nom de fichier
df["camera"] = df["filename"].str.extract(r"^([a-zA-Z0-9]+)_", expand=False).str.lower()

print(f"\nSources : {df['source_type'].value_counts().to_dict()}")
print(f"Caméras : {df['camera'].value_counts().to_dict()}")

In [ ]:
# [LÉGER] — Réconciliation avec les fichiers réels sur disque
# Le flag 'downloaded' dans l'index est incomplet.
# On vérifie l'existence réelle des fichiers.

def check_file_exists(row):
    """Vérifie si le fichier image existe physiquement."""
    expected = RAW_DIR / str(int(row["year"])) / f"{int(row['month']):02d}" / row["filename"]
    return expected.exists()

# Test rapide sur échantillon
sample_check = df.sample(min(200, len(df)), random_state=42)
sample_check_result = sample_check.apply(check_file_exists, axis=1)
print(f"Échantillon de vérification ({len(sample_check)} entrées) :")
print(f"  Fichiers trouvés   : {sample_check_result.sum()}")
print(f"  Fichiers manquants : {(~sample_check_result).sum()}")

# Compter les fichiers réels sur disque par année
disk_counts = {}
for year in range(2014, 2019):
    year_dir = RAW_DIR / str(year)
    if year_dir.exists():
        count = sum(
            1 for m in range(1, 13)
            for f in (year_dir / f"{m:02d}").glob("*.[jJ][pP][gG]")
            if (year_dir / f"{m:02d}").exists()
        )
        disk_counts[year] = count

print(f"\nFichiers réels sur disque par année :")
for y, c in sorted(disk_counts.items()):
    idx_count = len(df[df["year"] == y])
    print(f"  {y} : {c:,} sur disque / {idx_count:,} dans l'index")

In [ ]:
# [LÉGER] — Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"manquantes": missing, "%": missing_pct})
print(missing_df[missing_df["manquantes"] > 0].to_string())

# Doublons par filename (icone vs image_publique pointent vers le même fichier)
dup_count = df.duplicated(subset=["filename", "year", "month"], keep=False).sum()
print(f"\nDoublons (même filename + année + mois) : {dup_count:,}")
print("→ Normal : les URLs icone et image_publique partagent le même nom de fichier.")

In [ ]:
# [LÉGER] — Comptages par année, mois, heure
# Dédupliquer pour compter les images uniques (pas les URLs)
df_unique = df.drop_duplicates(subset=["filename", "year", "month"]).copy()
print(f"Images uniques (filename dédupliqué) : {len(df_unique):,}")

print("\n=== Par année ===")
print(df_unique["year"].value_counts().sort_index().to_string())

print("\n=== Par mois (toutes années) ===")
print(df_unique["month"].value_counts().sort_index().to_string())

print("\n=== Par heure (si disponible) ===")
hour_counts = df_unique["hour"].dropna().astype(int).value_counts().sort_index()
if len(hour_counts) > 0:
    print(hour_counts.to_string())
else:
    print("Pas d'information horaire disponible.")

print("\n=== Par caméra ===")
print(df_unique["camera"].value_counts().to_string())

---
## B. Couverture temporelle
Visualisations sur le CSV dédupliqué — **toutes légères**.

In [ ]:
# [LÉGER] — Barplot : images par année
fig, ax = plt.subplots(figsize=(8, 4))
yearly = df_unique["year"].value_counts().sort_index()
yearly.plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Nombre d'images uniques par année")
ax.set_xlabel("Année")
ax.set_ylabel("Nombre d'images")
for i, v in enumerate(yearly.values):
    ax.text(i, v + 20, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# [LÉGER] — Barplot : images par mois-année
df_unique["year_month"] = (
    df_unique["year"].astype(str) + "-" +
    df_unique["month"].apply(lambda m: f"{int(m):02d}")
)
monthly = df_unique["year_month"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(16, 5))
monthly.plot.bar(ax=ax, color="steelblue", edgecolor="white", width=0.8)
ax.set_title("Images uniques par mois-année (2014–2018)")
ax.set_xlabel("")
ax.set_ylabel("Nombre d'images")
# Afficher 1 label sur 3 pour la lisibilité
tick_labels = [label if i % 3 == 0 else "" for i, label in enumerate(monthly.index)]
ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# [LÉGER] — Heatmap de couverture temporelle
pivot = df_unique.groupby(["year", "month"]).size().unstack(fill_value=0)
# S'assurer que tous les mois sont représentés
for m in range(1, 13):
    if m not in pivot.columns:
        pivot[m] = 0
pivot = pivot[sorted(pivot.columns)]
pivot.columns = [f"{m:02d}" for m in pivot.columns]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    pivot, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5,
    ax=ax, cbar_kws={"label": "Nombre d'images"}
)
ax.set_title("Couverture temporelle — Images par mois et par année")
ax.set_xlabel("Mois")
ax.set_ylabel("Année")
plt.tight_layout()
plt.show()

In [ ]:
# [LÉGER] — Distribution horaire
hours = df_unique["hour"].dropna().astype(int)

if len(hours) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Histogramme horaire
    hours.value_counts().sort_index().plot.bar(
        ax=axes[0], color="teal", edgecolor="white"
    )
    axes[0].set_title("Distribution horaire des acquisitions")
    axes[0].set_xlabel("Heure (UTC)")
    axes[0].set_ylabel("Nombre d'images")

    # Jour vs Nuit (heuristique : 6h–18h = jour)
    df_unique["period"] = df_unique["hour"].apply(
        lambda h: "jour (6-18h)" if pd.notna(h) and 6 <= int(h) <= 18
        else "nuit (19-5h)"
    )
    df_unique["period"].value_counts().plot.pie(
        ax=axes[1], autopct="%1.1f%%", startangle=90,
        colors=["#f0c040", "#2c3e50"]
    )
    axes[1].set_title("Répartition jour / nuit")
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()
else:
    print("Pas de données horaires disponibles.")

In [ ]:
# [LÉGER] — Détection des trous temporels
all_months = set()
for y in range(2014, 2019):
    for m in range(1, 13):
        all_months.add((y, m))

covered_months = set(
    zip(df_unique["year"].astype(int), df_unique["month"].astype(int))
)
missing_months = sorted(all_months - covered_months)

if missing_months:
    print("Mois sans aucune image dans l'index :")
    for y, m in missing_months:
        print(f"  {y}-{m:02d}")
else:
    print("Tous les mois 2014–2018 sont couverts dans l'index.")

# Fréquence moyenne d'acquisition
daily_counts = df_unique.groupby(["year", "month", "day"]).size()
print(f"\nFréquence d'acquisition :")
print(f"  Médiane par jour : {daily_counts.median():.0f} images")
print(f"  Moyenne par jour : {daily_counts.mean():.1f} images")
print(f"  Max par jour     : {daily_counts.max()} images")
print(f"  Jours couverts   : {len(daily_counts)}")

# Caméras par année
print("\nCaméras actives par année :")
for y in range(2014, 2019):
    cams = df_unique[df_unique["year"] == y]["camera"].dropna().unique()
    print(f"  {y} : {', '.join(sorted(cams))}")

---
## C. Inspection visuelle (échantillonnage contrôlé)

**Stratégie** : on ne charge jamais plus de 16 images à la fois.  
On échantillonne par année, puis par période (jour/nuit).

In [ ]:
# [LÉGER] — Fonctions utilitaires d'affichage

def load_image_safe(filepath, max_size_mb=5):
    """Charge une image avec garde-fous."""
    fp = Path(filepath)
    if not fp.exists():
        return None
    if fp.stat().st_size > max_size_mb * 1024 * 1024:
        return None
    try:
        img = Image.open(fp)
        img.load()
        return img
    except Exception:
        return None


def show_image_grid(image_paths, titles=None, ncols=4, figsize_per_img=3):
    """Affiche une grille d'images. Charge une par une."""
    n = len(image_paths)
    if n == 0:
        print("Aucune image à afficher.")
        return
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_img * ncols, figsize_per_img * nrows)
    )
    axes = np.array(axes).flatten() if n > 1 else [axes]

    for i, ax in enumerate(axes):
        if i < n:
            img = load_image_safe(image_paths[i])
            if img is not None:
                ax.imshow(img)
                title = titles[i] if titles else Path(image_paths[i]).name
                ax.set_title(title, fontsize=7)
            else:
                ax.text(0.5, 0.5, "Introuvable", ha="center", va="center")
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def get_existing_image_paths(df_sub, n=12, seed=42):
    """Retourne au plus n chemins d'images existantes sur disque."""
    paths = []
    for _, row in df_sub.sample(frac=1, random_state=seed).iterrows():
        p = RAW_DIR / str(int(row["year"])) / f"{int(row['month']):02d}" / row["filename"]
        if p.exists():
            paths.append(str(p))
        if len(paths) >= n:
            break
    return paths

print("Fonctions d'affichage prêtes.")

In [ ]:
# [LÉGER] — Échantillon global : 12 images aléatoires
print("Échantillon aléatoire global (12 images) :")
sample_paths = get_existing_image_paths(df_unique, n=12, seed=42)
print(f"  {len(sample_paths)} images trouvées sur disque")
show_image_grid(sample_paths, ncols=4)

In [ ]:
# [LÉGER] — Échantillon par année (3 images par année)
for year in range(2014, 2019):
    df_y = df_unique[df_unique["year"] == year]
    if df_y.empty:
        continue
    paths = get_existing_image_paths(df_y, n=3, seed=year)
    if paths:
        print(f"\n--- {year} ({len(df_y)} images indexées, {len(paths)} affichées) ---")
        show_image_grid(paths, ncols=3, figsize_per_img=3)

In [ ]:
# [LÉGER] — Comparaison jour / nuit
if "period" in df_unique.columns:
    for period_name in ["jour (6-18h)", "nuit (19-5h)"]:
        df_p = df_unique[df_unique["period"] == period_name]
        paths = get_existing_image_paths(df_p, n=6, seed=123)
        if paths:
            print(f"\n--- {period_name.upper()} ({len(df_p)} images indexées) ---")
            show_image_grid(paths, ncols=3, figsize_per_img=3)
else:
    print("Pas de classification jour/nuit disponible (pas d'info horaire).")

In [ ]:
# [LÉGER] — Comparaison par caméra
top_cameras = df_unique["camera"].value_counts().head(3).index.tolist()
for cam in top_cameras:
    df_c = df_unique[df_unique["camera"] == cam]
    paths = get_existing_image_paths(df_c, n=4, seed=456)
    if paths:
        print(f"\n--- Caméra : {cam} ({len(df_c)} images) ---")
        show_image_grid(paths, ncols=4, figsize_per_img=3)

---
## D. Statistiques image (échantillon contrôlé)

**Stratégie** : calcul sur un échantillon stratifié par année,  
plafond absolu de 500 images. Environ 30 secondes.

In [ ]:
# [MOYEN] — Calcul de statistiques image (~30 sec pour 500 images)

MAX_SAMPLE = 500
PER_YEAR = 100

def compute_image_stats(filepath):
    """Calcule les statistiques d'une image."""
    fp = Path(filepath)
    if not fp.exists():
        return None
    try:
        file_size = fp.stat().st_size
        img = Image.open(fp)
        w, h = img.size
        arr = np.array(img.convert("L"), dtype=np.float32)
        img.close()
        return {
            "filepath": str(fp),
            "width": w,
            "height": h,
            "file_size_kb": file_size / 1024,
            "mean_brightness": float(arr.mean()),
            "std_brightness": float(arr.std()),
            "variance": float(arr.var()),
            "min_pixel": float(arr.min()),
            "max_pixel": float(arr.max()),
            "contrast": float(arr.max() - arr.min()),
        }
    except Exception:
        return None

# Échantillonnage stratifié par année
sample_rows = []
for year in range(2014, 2019):
    df_y = df_unique[df_unique["year"] == year]
    n_pick = min(PER_YEAR, len(df_y))
    sample_rows.append(df_y.sample(n_pick, random_state=42))

df_sample = pd.concat(sample_rows).head(MAX_SAMPLE)
print(f"Échantillon stratifié : {len(df_sample)} images")

# Résoudre les chemins et ne garder que ceux qui existent
sample_paths = []
sample_meta = []
for _, row in df_sample.iterrows():
    p = RAW_DIR / str(int(row["year"])) / f"{int(row['month']):02d}" / row["filename"]
    if p.exists():
        sample_paths.append(str(p))
        sample_meta.append({
            "year": int(row["year"]),
            "month": int(row["month"]),
            "camera": row.get("camera", ""),
            "hour": row.get("hour", None),
        })

print(f"Images existantes sur disque : {len(sample_paths)}")

# Calcul avec barre de progression
stats_list = []
for path, meta in tqdm(
    zip(sample_paths, sample_meta),
    total=len(sample_paths),
    desc="Statistiques images",
):
    s = compute_image_stats(path)
    if s:
        s.update(meta)
        stats_list.append(s)

df_stats = pd.DataFrame(stats_list)
print(f"\nStatistiques calculées pour {len(df_stats)} images.")
df_stats.describe().round(1)

In [ ]:
# [LÉGER] — Résolutions trouvées
print("Résolutions trouvées :")
res_counts = df_stats.groupby(["width", "height"]).size().sort_values(ascending=False)
for (w, h), count in res_counts.items():
    pct = count / len(df_stats) * 100
    print(f"  {w} x {h} : {count} images ({pct:.1f}%)")

print(f"\nTaille fichier : {df_stats['file_size_kb'].mean():.1f} KB (moyenne), "
      f"{df_stats['file_size_kb'].median():.1f} KB (médiane)")

In [ ]:
# [LÉGER] — Distributions de luminosité, variance, contraste, taille
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df_stats["mean_brightness"], bins=50, color="steelblue", edgecolor="white")
axes[0, 0].axvline(30, color="red", ls="--", label="Seuil nuit (30)")
axes[0, 0].set_title("Luminosité moyenne")
axes[0, 0].set_xlabel("Luminosité [0–255]")
axes[0, 0].legend()

axes[0, 1].hist(df_stats["variance"], bins=50, color="teal", edgecolor="white")
axes[0, 1].axvline(50, color="red", ls="--", label="Seuil nuageux (50)")
axes[0, 1].set_title("Variance")
axes[0, 1].set_xlabel("Variance")
axes[0, 1].legend()

axes[1, 0].hist(df_stats["contrast"], bins=50, color="coral", edgecolor="white")
axes[1, 0].set_title("Contraste (max - min)")
axes[1, 0].set_xlabel("Contraste")

axes[1, 1].hist(df_stats["file_size_kb"], bins=50, color="mediumpurple", edgecolor="white")
axes[1, 1].set_title("Taille de fichier")
axes[1, 1].set_xlabel("Taille (KB)")

plt.tight_layout()
plt.show()

In [ ]:
# [LÉGER] — Luminosité par année et classification jour/nuit
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df_stats, x="year", y="mean_brightness", ax=axes[0], palette="viridis")
axes[0].axhline(30, color="red", ls="--", alpha=0.7, label="Seuil nuit")
axes[0].set_title("Luminosité moyenne par année")
axes[0].legend()

df_stats["is_night_est"] = df_stats["mean_brightness"] < 30
night_pct = df_stats["is_night_est"].mean() * 100
day_pct = 100 - night_pct
axes[1].pie(
    [day_pct, night_pct],
    labels=[f"Jour ({day_pct:.1f}%)", f"Nuit ({night_pct:.1f}%)"],
    colors=["#f0c040", "#2c3e50"],
    autopct="%1.1f%%", startangle=90,
)
axes[1].set_title("Classification jour/nuit par luminosité")

plt.tight_layout()
plt.show()

In [ ]:
# [LÉGER] — Histogrammes de luminosité : 4 cas typiques
cases = {
    "Très sombre (nuit)": df_stats.nsmallest(1, "mean_brightness").iloc[0],
    "Sombre (aube)": df_stats.iloc[
        (df_stats["mean_brightness"] - 40).abs().argsort().iloc[0]
    ],
    "Jour couvert": df_stats.iloc[
        (df_stats["mean_brightness"] - 100).abs().argsort().iloc[0]
    ],
    "Jour clair": df_stats.nlargest(1, "mean_brightness").iloc[0],
}

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i, (label, row) in enumerate(cases.items()):
    img = load_image_safe(row["filepath"])
    if img is not None:
        axes[0, i].imshow(img)
        axes[0, i].set_title(label, fontsize=9)
        axes[0, i].axis("off")

        arr = np.array(img.convert("L")).flatten()
        axes[1, i].hist(arr, bins=64, color="gray", edgecolor="white", density=True)
        axes[1, i].axvline(row["mean_brightness"], color="red", ls="--")
        axes[1, i].set_title(
            f"mu={row['mean_brightness']:.0f}, sigma={row['std_brightness']:.0f}",
            fontsize=8,
        )
        axes[1, i].set_xlim(0, 255)
    else:
        axes[0, i].axis("off")
        axes[1, i].axis("off")

plt.suptitle("Histogrammes de luminosité — 4 cas typiques", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# [LÉGER] — Scatter luminosité vs variance (classification qualité)
fig, ax = plt.subplots(figsize=(10, 6))

conditions = [
    (df_stats["mean_brightness"] < 30),
    (df_stats["variance"] < 50),
]
labels_qf = ["dark", "cloudy"]
df_stats["quality_est"] = np.select(conditions, labels_qf, default="usable")

colors_map = {"dark": "#2c3e50", "cloudy": "#bdc3c7", "usable": "#27ae60"}
for qf, color in colors_map.items():
    mask = df_stats["quality_est"] == qf
    ax.scatter(
        df_stats.loc[mask, "mean_brightness"],
        df_stats.loc[mask, "variance"],
        c=color, label=f"{qf} (n={mask.sum()})",
        alpha=0.6, s=20,
    )

ax.axvline(30, color="red", ls="--", alpha=0.5, label="seuil nuit")
ax.axhline(50, color="orange", ls="--", alpha=0.5, label="seuil nuageux")
ax.set_xlabel("Luminosité moyenne")
ax.set_ylabel("Variance")
ax.set_title("Espace luminosité–variance : classification qualité estimée")
ax.legend()
plt.tight_layout()
plt.show()

print("\nRépartition qualité estimée :")
print(df_stats["quality_est"].value_counts().to_string())

---
## E. Observations scientifiques et implications

Résumé des constats exploratoires avant Phase 3.

In [ ]:
# [LÉGER] — Résumé quantitatif automatique
print("=" * 60)
print("RÉSUMÉ EDA — MERAPI 2014–2018")
print("=" * 60)

n_unique = len(df_unique)
n_disk = sum(disk_counts.values())
n_dark = (df_stats["quality_est"] == "dark").sum()
n_cloudy = (df_stats["quality_est"] == "cloudy").sum()
n_usable = (df_stats["quality_est"] == "usable").sum()
n_stats = len(df_stats)

res_mode = res_counts.idxmax()

print(f"""
1. VOLUME
   - Index total 2014-2018 : {n_unique:,} images uniques
   - Fichiers sur disque   : {n_disk:,}
   - Résolution dominante  : {res_mode[0]}x{res_mode[1]} (vignettes)
   - Taille fichier moy.   : {df_stats['file_size_kb'].mean():.1f} KB

2. QUALITÉ ESTIMÉE (échantillon n={n_stats})
   - Utilisables  : {n_usable} ({n_usable/n_stats*100:.1f}%)
   - Sombres/nuit : {n_dark} ({n_dark/n_stats*100:.1f}%)
   - Nuageuses    : {n_cloudy} ({n_cloudy/n_stats*100:.1f}%)

3. COUVERTURE TEMPORELLE
   - Mois manquants  : {len(missing_months)}
   - Images/jour moy : {daily_counts.mean():.1f}
   - Caméras         : {', '.join(sorted(df_unique['camera'].dropna().unique()))}

4. RÉSOLUTION
   Les images sur disque sont des VIGNETTES ({res_mode[0]}x{res_mode[1]}).
   Les images pleine résolution n'ont pas été téléchargées.
   Le redimensionnement 256x256 sera un upscaling.
""")

print("5. IMPLICATIONS POUR LA SUITE")
print("   - Phase 3 : le prétraitement doit séparer jour/nuit")
print("   - Les images nocturnes sont inutilisables en visuel")
print("     mais pourraient capturer des anomalies thermiques")
print("   - Les seuils qualité (luminosité=30, variance=50)")
print("     devront être validés sur l'ensemble complet")
print("   - Considérer le téléchargement des images pleine")
print("     résolution si les vignettes sont insuffisantes")
print("   - La météo (nuages, pluie) est un facteur dominant")
print("     de qualité des images de surveillance volcanique")

### Observations détaillées

**Différences jour / nuit** :
- Les images nocturnes (luminosité < 30) représentent une fraction significative du dataset.
- En vision optique, elles sont quasi-noires. Mais en contexte volcanique, des anomalies lumineuses (coulées, incandescence) pourraient être détectées même sur ces images.
- **Recommandation** : traiter jour et nuit comme deux sous-populations distinctes.

**Influence de la météo** :
- Les images à faible variance (< 50) mais luminosité > 30 sont typiquement nuageuses ou brumeuses.
- Le volcan Merapi est situé en zone tropicale (Java) : couverture nuageuse fréquente, surtout en saison humide (oct–mars).
- **Recommandation** : flag `cloudy` dans le quality filter, vérifier la saisonnalité.

**Images peu exploitables** :
- Vignettes 116x80 : résolution très faible pour de la détection fine.
- Images corrompues / trop petites : à filtrer (< 1 KB).
- Images uniformément noires ou blanches (variance ≈ 0) : inutilisables.

**Limites du dataset** :
- La résolution est celle de vignettes, pas d'images sources.
- L'index contient des doublons (icones + images_publiques → même fichier local).
- La couverture 2014 est limitée (< 500 images) vs 2016-2018 (> 3000/an).

**Prochaines étapes** :
1. **Phase 3a** — Prétraitement : redimensionnement, normalisation, séparation jour/nuit
2. **Phase 3b** — Filtrage qualité : classification automatique usable/dark/cloudy/corrupted
3. **Phase 3c** — Mise à jour de l'index avec les flags qualité
4. Envisager le téléchargement des images pleine résolution (images_publiques)

# 02 — Analyse exploratoire (EDA)
## Merapi Anomaly Detection — Phase 2

Ce notebook réalise l'**analyse exploratoire complète** du dataset d'images volcaniques Merapi.

### Objectifs
1. Comprendre la couverture temporelle des données
2. Analyser la distribution des images (heure, jour, mois)
3. Détecter les trous temporels et irrégularités
4. Quantifier la qualité initiale des images
5. Réfléchir aux biais de collecte (jour/nuit, météo, variabilité)

> **Note** : ce notebook fonctionne avec l'index réel (`index.csv`) ou de démonstration (`index_demo.csv`).
> Lancez d'abord `test_phase1.py` pour générer l'index réel.


## 0. Imports et configuration

In [ ]:
import sys
from pathlib import Path

# Racine du projet
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image

from src.utils import load_config, setup_logger, get_index_path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 4)})

config = load_config()
setup_logger(config)

FIGURES_DIR = PROJECT_ROOT / config["paths"]["figures"]
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration chargée —", config["project"]["name"])


## 1. Chargement de l'index

In [ ]:
index_real = get_index_path(config)
index_demo = PROJECT_ROOT / "data" / "index" / "index_demo.csv"

if index_real.exists():
    df = pd.read_csv(index_real, dtype=str, na_values=["", "None", "nan"])
    SOURCE = "réel"
    print(f"Index réel chargé : {len(df)} images")
else:
    df = pd.read_csv(index_demo, dtype=str, na_values=["", "None", "nan"])
    SOURCE = "démo"
    print(f"Index démo chargé : {len(df)} images (lancez test_phase1.py pour les vraies données)")

# Conversion de types de base
for c in ["year", "month", "day", "hour", "minute"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["downloaded"] = df["downloaded"].map(lambda x: str(x).lower() == "true")
df["file_size_bytes"] = pd.to_numeric(df["file_size_bytes"], errors="coerce")

# Fallback: extraire les composantes temporelles depuis filename si elles manquent
if "filename" in df.columns:
    parsed = df["filename"].str.extract(
        r"(?P<year_f>\d{4})-(?P<month_f>\d{2})-(?P<day_f>\d{2})_(?P<hour_f>\d{2})\.(?P<minute_f>\d{2})(?:\.\d{2})?"
    )
    for c in ["year", "month", "day", "hour", "minute"]:
        from_file = pd.to_numeric(parsed[f"{c}_f"], errors="coerce")
        if c not in df.columns:
            df[c] = from_file
        else:
            df[c] = df[c].fillna(from_file)

# to_datetime robuste aux NaN
dt_parts = df[["year", "month", "day", "hour", "minute"]].astype("float64")
df["datetime"] = pd.to_datetime(dt_parts, errors="coerce")

print("Colonnes disponibles :", list(df.columns))
print(f"Lignes avec datetime valide : {df['datetime'].notna().sum()}/{len(df)}")
df.head(3)


## 2. Statistiques descriptives globales

In [ ]:
print("=" * 55)
print(f"DATASET MERAPI — {SOURCE.upper()}")
print("=" * 55)
print(f"Total images indexées    : {len(df):,}")
print(f"Images téléchargées      : {df['downloaded'].sum():,}")
print(f"Années couvertes         : {sorted(df['year'].dropna().unique().tolist())}")
print(f"Mois distincts           : {df.groupby(['year','month']).ngroups}")
print(f"Plage temporelle         : {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Durée totale             : {(df['datetime'].max() - df['datetime'].min()).days} jours")
print()
print("Distribution qualité :")
print(df["quality_flag"].value_counts(dropna=False).to_string())
print()
print("Taille fichiers (octets) :")
print(df["file_size_bytes"].describe().apply(lambda x: f"{x:,.0f}").to_string())


## 3. Couverture temporelle

### 3.1 Images par mois

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

monthly = (
    df.groupby(["year", "month"])
    .size()
    .reset_index(name="count")
)
monthly["period"] = monthly.apply(lambda r: f"{int(r.year)}-{int(r.month):02d}", axis=1)

palette = sns.color_palette("Blues_d", len(monthly))
ax.bar(monthly["period"], monthly["count"], color=palette)
ax.set_xlabel("Période (année-mois)")
ax.set_ylabel("Nombre d'images")
ax.set_title("Couverture mensuelle — images indexées")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_coverage_monthly.png", bbox_inches="tight")
plt.show()

print(monthly[["period", "count"]].to_string(index=False))


### 3.2 Heatmap calendrier — présence journalière

In [ ]:
pivot = df.pivot_table(
    index=["year", "month"],
    columns="day",
    values="url",
    aggfunc="count",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(20, max(3, len(pivot) * 0.65)))
sns.heatmap(
    pivot, ax=ax, cmap="YlOrRd",
    linewidths=0.3, linecolor="white",
    cbar_kws={"label": "Nb images"},
    vmin=0,
)
ax.set_title("Présence journalière des images (heatmap calendrier)")
ax.set_xlabel("Jour du mois")
ax.set_ylabel("Année / Mois")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_calendar_heatmap.png", bbox_inches="tight")
plt.show()


### 3.3 Détection des trous temporels

In [ ]:
if df["datetime"].notna().any():
    dt_min = df["datetime"].min().date()
    dt_max = df["datetime"].max().date()
    dt_range = pd.date_range(dt_min, dt_max, freq="D")
    days_with_images = set(df["datetime"].dropna().dt.date)
    missing_days = [d.date() for d in dt_range if d.date() not in days_with_images]

    print(f"Jours couverts    : {len(days_with_images)}")
    print(f"Jours sans images : {len(missing_days)}")
    print(f"Taux de couverture: {len(days_with_images)/len(dt_range)*100:.1f}%")

    if missing_days:
        print(f"\nPremiers jours sans images :")
        for d in missing_days[:10]:
            print(f"  {d}")
        if len(missing_days) > 10:
            print(f"  ... et {len(missing_days)-10} autres jours manquants.")
else:
    print("Champ 'datetime' non disponible — vérifiez le format de l'index.")


## 4. Distribution intra-journalière

### 4.1 Images par heure (cycle diurne)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hourly_all = df.groupby("hour").size()
axes[0].bar(hourly_all.index.astype(int), hourly_all.values, color="#4878CF")
axes[0].set_title("Toutes images — distribution horaire")
axes[0].set_xlabel("Heure (UTC)  |  heure locale = UTC+7")
axes[0].set_ylabel("Nombre d'images")
axes[0].set_xticks(range(0, 24))

df_usable = df[df["quality_flag"] == "usable"]
if not df_usable.empty:
    hourly_usable = df_usable.groupby("hour").size()
    axes[1].bar(hourly_usable.index.astype(int), hourly_usable.values, color="#6ACC65")
    axes[1].set_title("Images utilisables — distribution horaire")
    axes[1].set_xlabel("Heure (UTC)")
    axes[1].set_xticks(range(0, 24))
else:
    axes[1].text(
        0.5, 0.5,
        "Pas d'images 'usable'\n(prétraitement Phase 3 non encore effectué)",
        ha="center", va="center", transform=axes[1].transAxes, color="gray"
    )

plt.suptitle("Cycle diurne d'acquisition", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_hourly_distribution.png", bbox_inches="tight")
plt.show()

print("Note : Merapi (Java) est à UTC+7.")
print("  Midi local ≈ 05h UTC  |  minuit local ≈ 17h UTC")


### 4.2 Proportion jour / nuit

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

if "is_night" in df.columns and df["is_night"].notna().any():
    night_map = {"True": "Nuit", "False": "Jour", True: "Nuit", False: "Jour"}
    night_counts = df["is_night"].map(night_map).value_counts()
else:
    df["_is_night_computed"] = (df["hour"] < 6) | (df["hour"] >= 19)
    night_counts = df["_is_night_computed"].map({True: "Nuit", False: "Jour"}).value_counts()

ax.pie(
    night_counts.values,
    labels=night_counts.index,
    autopct="%1.1f%%",
    colors=["#2B3A67", "#F2A65A"],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
ax.set_title("Proportion images Jour / Nuit")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_day_night_ratio.png", bbox_inches="tight")
plt.show()

print("Remarque : les images nocturnes ne sont PAS inutilisables par défaut.")
print("Elles sont précieuses pour détecter l'incandescence et les explosions nocturnes.")
print("→ Elles nécessitent un pipeline de traitement adapté (normalisation distincte).")


## 5. Analyse de la qualité des images

In [ ]:
COLORS_Q = {
    "usable": "#6ACC65", "cloudy": "#9B9B9B", "dark": "#2B3A67",
    "corrupted": "#E24B4A", "unknown": "#FAC775",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# -- Graphique 1 : distribution globale ---------------------
quality_counts = df["quality_flag"].fillna("unknown").value_counts()
bar_colors = [COLORS_Q.get(str(q), "#CCCCCC") for q in quality_counts.index]
axes[0].bar(quality_counts.index.astype(str), quality_counts.values, color=bar_colors)
axes[0].set_title("Distribution globale des qualités")
axes[0].set_xlabel("Quality flag")
axes[0].set_ylabel("Nombre d'images")

# -- Graphique 2 : qualité par mois (empilée) ---------------
df_q = df.copy()
df_q["quality_flag"] = df_q["quality_flag"].fillna("unknown")
# Forcer year/month en int pour le groupby (évite float après to_numeric)
df_q["year"] = df_q["year"].astype("Int64")
df_q["month"] = df_q["month"].astype("Int64")

monthly_q = (
    df_q.groupby(["year", "month", "quality_flag"])
    .size()
    .unstack(fill_value=0)
)
# Garantir que toutes les valeurs sont bien numériques (int)
monthly_q = monthly_q.apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
monthly_q.index = [f"{int(y)}-{int(m):02d}" for y, m in monthly_q.index]

if monthly_q.empty:
    axes[1].text(0.5, 0.5, "Pas de données\n(quality_flag entièrement NaN)",
                 ha="center", va="center", transform=axes[1].transAxes, color="gray")
else:
    monthly_q.plot(
        kind="bar", stacked=True, ax=axes[1],
        color=[COLORS_Q.get(c, "#CCCCCC") for c in monthly_q.columns],
    )
    axes[1].legend(title="Qualité", bbox_to_anchor=(1.01, 1), loc="upper left")

axes[1].set_title("Qualité par mois (empilée)")
axes[1].set_xlabel("Période")
axes[1].set_ylabel("Nombre d'images")
plt.xticks(rotation=45, ha="right")

plt.suptitle("Qualité des images — vue globale et temporelle", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "05_quality_distribution.png", bbox_inches="tight")
plt.show()


## 6. Distribution des tailles de fichiers

In [ ]:
df_dl = df[df["downloaded"] & df["file_size_bytes"].notna()].copy()

if df_dl.empty:
    print("Aucune image téléchargée.")
    print("→ Exécutez : python test_phase1.py --max-images 50")
else:
    df_dl["size_kb"] = df_dl["file_size_bytes"] / 1024

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(df_dl["size_kb"], bins=40, color="#4878CF", edgecolor="white")
    axes[0].set_title("Distribution des tailles de fichiers")
    axes[0].set_xlabel("Taille (Ko)")
    axes[0].set_ylabel("Fréquence")
    median_kb = df_dl["size_kb"].median()
    axes[0].axvline(median_kb, color="red", linestyle="--",
                    label=f"Médiane : {median_kb:.0f} Ko")
    axes[0].legend()

    for quality, grp in df_dl.groupby("quality_flag"):
        axes[1].hist(grp["size_kb"], bins=30, alpha=0.65, label=str(quality))
    axes[1].set_title("Taille des fichiers par qualité")
    axes[1].set_xlabel("Taille (Ko)")
    axes[1].legend()

    plt.suptitle("Tailles de fichiers (indicateur indirect de contenu)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "06_file_sizes.png", bbox_inches="tight")
    plt.show()

    print("Les images très petites (< 10 Ko) sont souvent corrompues ou noires.")
    print("Les images nocturnes sont souvent plus compactes (moins d'information).")


## 7. Visualisation d'échantillons d'images

In [ ]:
def show_image_sample(df_subset, title, n=5, figsize=(15, 4)):
    """Affiche un échantillon d'images depuis les chemins locaux."""
    rows_dl = df_subset[df_subset["downloaded"] == True]
    if rows_dl.empty:
        print(f"[{title}] Aucune image téléchargée dans ce sous-ensemble.")
        return
    rows = rows_dl.sample(min(n, len(rows_dl)), random_state=42)

    fig, axes = plt.subplots(1, len(rows), figsize=figsize)
    if len(rows) == 1:
        axes = [axes]
    fig.suptitle(title, fontweight="bold")

    for ax, (_, row) in zip(axes, rows.iterrows()):
        local_path = PROJECT_ROOT / str(row["local_path"])
        if local_path.exists():
            try:
                img = Image.open(local_path)
                ax.imshow(img, cmap="gray" if img.mode == "L" else None)
                ax.set_title(f"J={row.get('day','?')} H={row.get('hour','?')}h", fontsize=8)
            except Exception as e:
                ax.text(0.5, 0.5, f"Erreur:\n{e}", ha="center", va="center",
                        transform=ax.transAxes, fontsize=7, color="red")
        else:
            ax.text(0.5, 0.5, "Fichier\nlocal\nmanquant", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="gray")
        ax.axis("off")

    plt.tight_layout()
    safe_name = title.lower().replace(" ", "_").replace(":", "")[:40]
    plt.savefig(FIGURES_DIR / f"07_sample_{safe_name}.png", bbox_inches="tight")
    plt.show()

for flag in ["usable", "cloudy", "dark"]:
    subset = df[df["quality_flag"] == flag]
    if not subset.empty:
        show_image_sample(subset, f"Échantillon — {flag}", n=5)
    else:
        print(f"Qualité '{flag}' : aucune entrée dans l'index.")


## 8. Série temporelle de luminosité

In [ ]:
def compute_image_stats(local_path):
    """Luminosité moyenne et variance d'une image (niveaux de gris)."""
    try:
        arr = np.array(Image.open(local_path).convert("L"), dtype=np.float32)
        return {
            "mean_brightness": float(arr.mean()),
            "std_brightness": float(arr.std()),
            "variance": float(arr.var()),
        }
    except Exception:
        return {"mean_brightness": None, "std_brightness": None, "variance": None}

df_dl = df[df["downloaded"] == True].copy()

if df_dl.empty:
    print("Aucune image téléchargée — section ignorée.")
    df_stats = pd.DataFrame()
else:
    stats_list = []
    for _, row in df_dl.iterrows():
        path = PROJECT_ROOT / str(row["local_path"])
        stats = compute_image_stats(path)
        stats.update({
            "url": row["url"],
            "datetime": row["datetime"],
            "hour": row["hour"],
            "quality_flag": row["quality_flag"],
        })
        stats_list.append(stats)

    df_stats = pd.DataFrame(stats_list).dropna(subset=["mean_brightness"])
    df_stats["datetime"] = pd.to_datetime(df_stats["datetime"])
    df_stats = df_stats.sort_values("datetime")
    print(f"Statistiques calculées sur {len(df_stats)} images téléchargées.")


In [ ]:
if "df_stats" in dir() and not df_stats.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    sc0 = axes[0].scatter(df_stats["datetime"], df_stats["mean_brightness"],
                          c=df_stats["mean_brightness"], cmap="plasma", s=10, alpha=0.7)
    axes[0].set_ylabel("Luminosité moyenne (0–255)")
    axes[0].set_title("Série temporelle — luminosité moyenne")
    plt.colorbar(sc0, ax=axes[0], label="Luminosité")

    sc1 = axes[1].scatter(df_stats["datetime"], df_stats["variance"],
                          c=df_stats["variance"], cmap="viridis", s=10, alpha=0.7)
    axes[1].set_ylabel("Variance (proxy netteté)")
    axes[1].set_title("Série temporelle — variance (proxy activité / qualité)")
    axes[1].set_xlabel("Date")
    plt.colorbar(sc1, ax=axes[1], label="Variance")

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "08_brightness_timeseries.png", bbox_inches="tight")
    plt.show()

    print("Pic de luminosité nocturne → candidat incandescence volcanique.")
    print("Variance < 50 (image diurne) → probablement nuageuse ou brumeuse.")
    print("Ces deux indicateurs constituent les premières BASELINES (Phase 4).")
else:
    print("Section ignorée — aucune donnée de statistiques disponible.")


## 9. Profil diurne moyen

In [ ]:
if "df_stats" in dir() and not df_stats.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    hourly = df_stats.groupby("hour")["mean_brightness"].agg(["mean", "std"]).reset_index()

    axes[0].fill_between(
        hourly["hour"],
        hourly["mean"] - hourly["std"],
        hourly["mean"] + hourly["std"],
        alpha=0.25, color="#4878CF",
    )
    axes[0].plot(hourly["hour"], hourly["mean"], "o-", color="#4878CF")
    axes[0].set_title("Profil diurne moyen — luminosité")
    axes[0].set_xlabel("Heure (UTC)  |  locale = UTC+7")
    axes[0].set_ylabel("Luminosité moyenne (0–255)")
    axes[0].set_xticks(range(0, 24))
    axes[0].axvspan(-0.5, 5.5, alpha=0.08, color="#2B3A67", label="Nuit (UTC)")
    axes[0].axvspan(18.5, 23.5, alpha=0.08, color="#2B3A67")
    axes[0].legend()

    hourly_var = df_stats.groupby("hour")["variance"].mean()
    axes[1].bar(hourly_var.index.astype(int), hourly_var.values, color="#F2A65A")
    axes[1].set_title("Variance moyenne par heure")
    axes[1].set_xlabel("Heure (UTC)")
    axes[1].set_ylabel("Variance moyenne")
    axes[1].set_xticks(range(0, 24))

    plt.suptitle("Cycle diurne — référence pour la détection (Phase 4)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "09_diurnal_cycle.png", bbox_inches="tight")
    plt.show()

    print("Ce profil diurne servira de RÉFÉRENCE en Phase 4.")
    print("Un écart fort par rapport à cette baseline horaire → anomalie candidate.")
else:
    print("Section ignorée — calculez d'abord les statistiques (cellule précédente).")


## 10. Bilan EDA — conclusions scientifiques

In [ ]:
n_total = len(df)
n_usable = (df["quality_flag"] == "usable").sum()
n_dark = (df["quality_flag"] == "dark").sum()
n_cloudy = (df["quality_flag"] == "cloudy").sum()
n_months = df.groupby(["year", "month"]).ngroups

print("=" * 65)
print("BILAN EDA — MERAPI ANOMALY DETECTION")
print("=" * 65)
print(f"""
COUVERTURE DES DONNÉES
  Total images indexées : {n_total:,}
  Mois couverts         : {n_months}
  Images utilisables    : {n_usable:,}  ({n_usable/n_total*100:.0f}%)
  Images nocturnes      : {n_dark:,}   ({n_dark/n_total*100:.0f}%)
  Images nuageuses      : {n_cloudy:,}   ({n_cloudy/n_total*100:.0f}%)

BIAIS PRINCIPAUX IDENTIFIÉS
  1. Cycle jour/nuit fort : la luminosité varie de 0 à ~200 niveaux.
     → Traiter les images jour/nuit séparément OU normaliser par heure.

  2. Nuages/météo : principal générateur de faux positifs.
     → Filtre de variance indispensable avant toute modélisation.

  3. Trous temporels : jours sans images à gérer explicitement.
     → Pas d'interpolation — signaler dans les scores d'anomalie.

  4. Variabilité saisonnière : à anticiper pour les modèles IA (Phase 5).

RECOMMANDATIONS POUR LA PHASE 3 (Prétraitement)
  ✓ Séparer explicitement corpus jour / nuit
  ✓ Filtrer les images inutilisables (variance < seuil config)
  ✓ Normaliser par profil horaire moyen (corriger le cycle diurne)
  ✓ Préparer la ROI autour du dôme volcanique (à définir avec volcanologues)
  ✓ Redimensionner à {config['preprocessing']['target_size']} px

RECOMMANDATIONS POUR LA PHASE 4 (Baselines)
  ✓ Variance et luminosité moyenne → signaux candidats directs
  ✓ Comparer à la même heure J-7 → meilleur contrôle du biais diurne
  ✓ SSIM entre images consécutives → détecte les changements brusques
  ✓ Score de luminosité nocturne → candidat incandescence
""")

print(f"Figures sauvegardées dans : {FIGURES_DIR}")
